In [ ]:
"""
You can run either this notebook locally or on Google Colab.

Instructions for setting up Colab are as follows:
1. Open a new Python 3 notebook.
2. Import this notebook from GitHub (File -> Upload Notebook -> "GITHUB" tab -> copy/paste GitHub URL)
3. Optional: Restart the runtime (Runtime -> Restart Runtime) for any upgraded packages to take effect
"""

# Building a Bantu language on the shared ITN kernel: Luganda

Inverse text normalization (ITN) turns what a speech recognizer hears into what a reader or
a downstream system needs: a Luganda speaker says *emitwalo ebiri*, and a payments pipeline
needs `20,000`.

This notebook shows how Luganda (`lg`) was built on NeMo's shared **Bantu ITN kernel**: what
the kernel does for every Bantu language, what Luganda supplies, how it is tested, and one
problem we got stuck on and how we worked through it. It is meant as a starting point for
anyone adding another Bantu language.

## Prerequisites
1. Read the [Text (Inverse) Normalization tutorial](https://colab.research.google.com/github/NVIDIA/NeMo-text-processing/blob/main/tutorials/Text_(Inverse)_Normalization.ipynb) first.
   The [WFST tutorial](https://github.com/NVIDIA/NeMo-text-processing/blob/main/tutorials/WFST_Tutorial.ipynb)
   explains the taggers, verbalizers and token format used below.
2. No Luganda is needed. Each example says what the words mean.

# Table of Contents
1. <a href='#setup'>Set up</a>
2. <a href='#two-examples'>Two Luganda examples</a>
3. <a href='#language-files'>The language files</a>
4. <a href='#reuse'>What the kernel does, what Luganda supplies</a>
5. <a href='#tests'>Tests</a>
6. <a href='#worked-example'>Worked example: one word, two numbers</a>
7. <a href='#next-contributor'>Changing a word list</a>
8. <a href='#your-language'>Adding your language</a>

# 1. Set up <a id="setup"></a>

Luganda is on the `staging/bantu_itn` branch of NeMo-text-processing. The install cell is
skipped if `nemo_text_processing` is already importable (for example, when you run the
notebook from a checkout).

In [ ]:
## Install NeMo-text-processing
REPO = "https://github.com/NVIDIA/NeMo-text-processing.git"
BRANCH = "staging/bantu_itn"

import importlib.util

if importlib.util.find_spec("nemo_text_processing") is None:
    %pip install "git+{REPO}@{BRANCH}#egg=nemo_text_processing"

In [ ]:
from dataclasses import replace

import pynini
from pynini.lib import rewrite

from nemo_text_processing.inverse_text_normalization.bantu.taggers.tokenize_and_classify import build_classifier
from nemo_text_processing.inverse_text_normalization.inverse_normalize import InverseNormalizer
from nemo_text_processing.inverse_text_normalization.lg.settings import DATA_DIR, load
from nemo_text_processing.inverse_text_normalization.lg.verbalizers.verbalize_final import VerbalizeFinalFst

itn = InverseNormalizer(lang="lg")
print("pynini", pynini.__version__)

# 2. Two Luganda examples <a id="two-examples"></a>

*Abiri* means "two" when it counts eggs (*amagi*), but after a currency phrase it is the short
form of "twenty":

In [ ]:
for spoken in ["amagi abiri", "ssiringi za uganda abiri"]:
    print(f"{spoken!r:30} -> {itn.inverse_normalize(spoken, verbose=False)!r}")

The same word gives two different numbers. **The word before a number decides its value.**
That is the problem the kernel was built around, and section 6 works through it.

NeMo's ITN runs in two steps: a **tagger** (classifier) turns the sentence into tokens, and a
**verbalizer** writes the tokens out. In the Bantu kernel the tagger resolves the value; the
verbalizer only formats it (digit grouping, `<amount> <ISO code>`):

In [ ]:
def tags(text, fst=None):
    """The tagger's best output for text."""
    return rewrite.top_rewrite(text, fst or itn.tagger.fst)


for spoken in ["amagi abiri", "ssiringi za uganda abiri"]:
    print(tags(spoken))

A few more, each with what it means:

| Spoken | Meaning |
|---|---|
| *abantu babiri* | two people (the number agrees with the noun: *ba-biri*) |
| *emiti nkumi bbiri mu bikumi bibiri mu amakumi abiri mu etaano* | trees: two thousand, two hundred, twenty and five |
| *yampa emitwalo ebiri* | he gave me two ten-thousands (money) |
| *omuti guli mu nnimiro* | the tree is in the garden (*mu* is "in" here, not a number join) |

In [ ]:
for spoken in [
    "abantu babiri",
    "emiti nkumi bbiri mu bikumi bibiri mu amakumi abiri mu etaano",
    "yampa emitwalo ebiri",
    "omuti guli mu nnimiro",
]:
    print(f"{spoken!r} -> {itn.inverse_normalize(spoken, verbose=False)!r}")

# 3. The language files <a id="language-files"></a>

A Bantu language is a folder like any NeMo language. Everything specific to Luganda is here:

```
nemo_text_processing/inverse_text_normalization/lg/
  settings.py                       the shape of the grammar; load() fills it from data/
  data/numbers/*.tsv                number words, places, scale words, roles, spelling variants
  data/money/*.tsv                  currency phrases, money cues
  taggers/tokenize_and_classify.py  ClassifyFst: calls the kernel's build_classifier
  verbalizers/verbalize.py          VerbalizeFst (for grammar export)
  verbalizers/verbalize_final.py    VerbalizeFinalFst
```

Registration is a few small edits: an `elif lang == 'lg'` branch and the CLI choice in
`inverse_normalize.py`, the choice in `run_evaluate.py`, and a branch in `pynini_export.py`
(for Sparrowhawk export).

In [ ]:
import os

for folder in ("numbers", "money"):
    for name in sorted(os.listdir(os.path.join(DATA_DIR, folder))):
        if name.endswith(".tsv"):
            with open(os.path.join(DATA_DIR, folder, name), encoding="utf-8") as f:
                rows = f.read().splitlines()
            print(f"data/{folder}/{name:22} {len(rows):3} rows   e.g. {rows[0].split(chr(9))}")

**`settings.py`** holds what is not a word list: the noun classes (one example noun each),
the word for ten, the join words, the places, and the counted scales with their switches.
`load()` fills it from `data/`:

In [ ]:
profile = load()
print("noun classes:", ", ".join(f"{c.class_id} ({c.noun})" for c in profile.contexts))
print("ten:", profile.ten, "| joins:", sorted(profile.join_words()))
print("scales:", ", ".join(f"{s.plural} (10^{s.power})" for s in profile.counted_scales))
print("largest value:", f"{profile.max_value:,}")

**Number words agree with the noun class.** "Two" has a different form for people, trees,
eggs and cows. Each row of `units.tsv` is *form, value, class, kind*:

In [ ]:
with open(os.path.join(DATA_DIR, "numbers", "units.tsv"), encoding="utf-8") as f:
    for row in f.read().splitlines():
        form, value, class_id, kind = row.split("\t")
        if value == "2" and kind == "any":
            noun = next(c.noun for c in profile.contexts if c.class_id == class_id)
            print(f"class {class_id:>2} ({noun:8}) two = {form}")

# 4. What the kernel does, what Luganda supplies <a id="reuse"></a>

| The kernel (shared by every Bantu language) | Luganda (this folder) |
|---|---|
| Marks runs of number words and reads each run in the context of the word before it | Which words are number words, per noun class |
| Composes place values (thousands, hundreds, tens, units) | The place words and the join words *na* and *mu* |
| Counted scales (*mitwalo* = ten-thousands) with their counts | The scale words and their switches |
| Money: an amount after a currency phrase or money cue | Currency phrases, money cues, the default currency (UGX) |
| Exactly one best reading for every input | Nothing: it is checked by the tests |
| | One Luganda step: split contracted *n'* ("and") off a scale word |

**One rule from the kernel:** a join word that is also an ordinary word only joins numbers
when a number follows it. *Mu* joins "twenty *mu* five", but in *omuti guli mu nnimiro* it
means "in" and the sentence passes through unchanged (section 2).

**One rule from Luganda:** Luganda writes "and" before a vowel attached to the next word,
*n'emitwalo*. Luganda's `ClassifyFst` offers a split (`ne mitwalo`) before the kernel reads
the sentence, and keeps it only where the kernel then converts the amount:

In [ ]:
for spoken in ["yampa emmere n'emitwalo ebiri", "n'omutwalo gwange"]:
    print(f"{spoken!r:34} -> {itn.inverse_normalize(spoken, verbose=False)!r}")

("He gave me food and twenty thousand", and "and my ten thousand", which has no amount and is
left as it is.) If a second language needs the same split, it moves into the kernel.

**What another language changes:** its own `settings.py` and `data/*.tsv`, and its tests. No
kernel code. Swahili is next, built the same way.

# 5. Tests <a id="tests"></a>

Luganda's tests are spoken/written pairs in `tests/nemo_text_processing/lg/data_inverse_text_normalization/`,
one file per class (cardinal, money, word), all reviewed by a native speaker. From a checkout:

```bash
pytest tests/nemo_text_processing/lg --cpu
bash tools/text_processing_deployment/export_grammars.sh --MODE=test --GRAMMARS=itn_grammars --LANGUAGE=lg
```

The second command exports the grammars and runs the same pairs through Sparrowhawk in Docker.

**Every test also checks that the input has exactly one best reading.** NeMo's runtime picks
one of several equally good readings silently, so a grammar can look right in a test and be
wrong after a rebuild or in Sparrowhawk. `optimal_rewrites` returns every best reading:

In [ ]:
def readings(text, fst=None):
    """All equally best tagger outputs. A Bantu language must give exactly one."""
    return set(rewrite.optimal_rewrites(text, fst or itn.tagger.fst))


cases = [
    ("amagi abiri", "amagi 2"),
    ("ssiringi za uganda abiri", "20 UGX"),
    ("abantu emitwalo abiri", "abantu 200,000"),
    ("yampa emitwalo ebiri", "yampa 20,000 UGX"),
    ("omuti guli mu nnimiro", "omuti guli mu nnimiro"),
]
for spoken, written in cases:
    assert len(readings(spoken)) == 1, spoken
    assert itn.inverse_normalize(spoken, verbose=False) == written, spoken
print(f"{len(cases)} cases: one best reading each, all as expected")

**Known limits are tested too.** A money word only counts when it comes right before the
amount, so in *ssente ziri emitwalo esatu* ("the money is thirty thousand") the amount converts
but without the currency:

In [ ]:
print(itn.inverse_normalize("ssente ziri emitwalo esatu", verbose=False))

# 6. Worked example: one word, two numbers <a id="worked-example"></a>

This is the problem the kernel was designed around, and the first place we got stuck.

**Input and expected output** (reviewed by a native speaker):

| Spoken | Written | Why |
|---|---|---|
| *amagi abiri* | `amagi 2` | *abiri* agrees with *amagi* (eggs, class 6): "two" |
| *ssiringi za uganda abiri* | `20 UGX` | after a currency phrase, *abiri* is the short form of "twenty" |

### What does not work: a word-to-number table

Most NeMo languages map number words to digits with a table such as `cardinal.tsv`. A table
can only give *abiri* one value, or both:

In [ ]:
table = pynini.string_map([("abiri", "2"), ("abiri", "20")])
print(readings("abiri", table))

Two equally good readings: a tie. NeMo would pick one silently, and whichever it picks is
wrong for half of the sentences. Weighting one reading higher does not help either: both
readings are right, in different contexts.

### What we tried: where should the context be read?

The value depends on the word *before* the number, so something has to look at both. We built
both options on the same Luganda data and compared them:

| | A: resolve in the tagger | B: resolve in the verbalizer |
|---|---|---|
| Tagger emits | a resolved `cardinal { integer: "2" }` | the raw text, `cardinal_candidate { text: "abiri" }` |
| Verbalizer | formats digits only | looks at the neighbouring noun, decides the value |
| Result on 131 reviewed cases | 131/131 | 131/131 |

Both were correct, so the choice came down to the design. **We chose A**, and the NeMo
maintainers confirmed it: the tokens stay standard NeMo tokens, every verbalizer stays
simple and the same for every language, and the tagger output already means something
(B also tagged the "in" sense of *mu* as a number candidate, leaving it to the verbalizer
to undo).

### The fix: roles, licensed by context

So the tagger reads each number in the context of the word before it. *Abiri* gets two
**roles** in `roles.tsv`, and only a matching context can use each one:

In [ ]:
with open(os.path.join(DATA_DIR, "numbers", "roles.tsv"), encoding="utf-8") as f:
    print(f.read())
with open(os.path.join(DATA_DIR, "money", "currency.tsv"), encoding="utf-8") as f:
    print("".join(row for row in f if row.startswith("ssiringi za uganda")))

- `counted-unit/class-6`: *abiri* = 2 when it counts a class-6 noun such as *amagi*.
- `shortened-tens`: *abiri* = 20, and only contexts that list this role may use it. The
  currency phrases list it (last column of `currency.tsv`).

**Why no weights are needed.** "Two" agreeing with *ssiringi* (shillings, class 10) is
*bbiri*, not *abiri*. So after a currency phrase, *abiri* can only be "twenty". Each context
allows exactly one reading, and the structure of the grammar makes the answer unique.

In [ ]:
for spoken in ["amagi abiri", "ssiringi za uganda abiri", "ssiringi za uganda bbiri"]:
    print(f"{spoken!r:28} readings: {len(readings(spoken))}   -> {itn.inverse_normalize(spoken, verbose=False)!r}")

### Proving the role does the work

Take the `shortened-tens` role away from the currency phrases and rebuild the tagger. The
currency context can no longer read *abiri* as twenty:

In [ ]:
verbalizer = VerbalizeFinalFst().fst


def normalize_with(profile, text):
    """Build a tagger from profile and run text through it and the verbalizer."""
    tokens = rewrite.top_rewrite(text, build_classifier(profile))
    return rewrite.top_rewrite(tokens, verbalizer)


without_role = replace(profile, currencies=tuple(replace(c, roles=()) for c in profile.currencies))
print("with the role:   ", normalize_with(profile, "ssiringi za uganda abiri"))
print("without the role:", normalize_with(without_role, "ssiringi za uganda abiri"))

Without the role, *abiri* after the currency phrase is not a number the currency can count,
so the sentence is left as spoken. That is safe (nothing is converted wrongly), but it is not
what a speaker means.

### The test that checks it

Both sentences are in the Luganda case files, and the test asserts the output and that there
is exactly one best reading:

```
test_cases_cardinal.txt:  amagi abiri~amagi 2
test_cases_money.txt:     ssiringi za uganda abiri~20 UGX
```

**What to take from this for your language:** when one word can mean two numbers, do not pick
one by weight. Find what in the context decides it (here, agreement: the currency noun's own
"two" is a different word), give the word one role per meaning, and let only the matching
context use each role.

### Other problems we hit

Each has a test in the Luganda or kernel case files:

- **Ties the runtime hides.** Every test checks for exactly one best reading
  (`optimal_rewrites`), which found a kernel tie: a join word at the start of a run.
- **Join words that are ordinary words** (*mu* = "in"): a join only joins when a number follows.
- **A number converted only in part** (*amagi kkumi n'abiri*, 12 eggs, gave `amagi 10 n'abiri`):
  the contracted teen forms were missing from the word lists.
- **A pre-step that rewrote text it did not convert:** the *n'* split first checked "is a
  number word next?" itself, and got it wrong. Now the kernel's own reading decides, and the
  split is kept only where an amount follows.
- **Natural forms come from the speaker.** Sentences we wrote were grammatical but not what a
  speaker says; every test pair was reviewed.

# 7. Changing a word list <a id="next-contributor"></a>

Most changes to a language are one row in a TSV. A real example: during review, the speaker
added *bampa* (a form of "give me") as a money word, so that the amount after it is read as UGX.

Before the change, without *bampa* in `cues.tsv`:

In [ ]:
cues = profile.money_cues
before = replace(profile, money_cues=replace(cues, words=tuple(w for w in cues.words if w != "bampa")))
print("before:", normalize_with(before, "bampa emitwalo ebiri"))
print("after: ", normalize_with(profile, "bampa emitwalo ebiri"))

The change itself, in a checkout:

1. Add the row to `nemo_text_processing/inverse_text_normalization/lg/data/money/cues.tsv`:
   ```
   bampa	word
   ```
   (`word` matches the whole word; `stem` would match it inside longer words too, which can
   catch unrelated words, so short forms are listed as whole words.)
2. Add the expected output to `tests/nemo_text_processing/lg/data_inverse_text_normalization/test_cases_money.txt`:
   ```
   bampa emitwalo ebiri~bampa 20,000 UGX
   ```
   and, for a new cue, a negative case: a similar word that must *not* be caught.
3. Rerun the tests: `pytest tests/nemo_text_processing/lg --cpu`.

Every language change is reviewed by a native speaker before it goes in.

# 8. Adding your language <a id="your-language"></a>

1. **Gather the paradigm with a native speaker:** number words for each noun class, tens,
   hundreds, thousands, the join words, scale words, currency phrases, and the sentences
   people actually say.
2. **Copy `lg/` (or the test language `toy/`)** to `inverse_text_normalization/<your code>/`,
   with `__init__.py` in every folder, including `data/`.
3. **Fill `settings.py` and `data/*.tsv`.** No kernel code.
4. **Register** the language in `InverseNormalizer`, the CLI choices, `run_evaluate.py` and
   `pynini_export.py`.
5. **Write the test pairs** (cardinal, money, word) and have the speaker review every row.
   Include sentences that must stay unchanged.
6. **Check one best reading** for every case, and run the Sparrowhawk tests.

Questions and new languages are welcome on the
[NeMo-text-processing repository](https://github.com/NVIDIA/NeMo-text-processing).